In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ---------------------------------------------------------
# Helper for keeping the latest version of each record
# ---------------------------------------------------------

def latest_record(df, business_key):
    window = (
        Window
        .partitionBy(business_key)
        .orderBy(
            F.col("updated_at").desc(),
            F.col("_airbyte_extracted_at").desc(),
        )
    )

    return (
        df
        .withColumn("_row_number", F.row_number().over(window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )


# ---------------------------------------------------------
# 1. Campaigns
# ---------------------------------------------------------

bronze_campaigns = spark.table(
    "voyage.bronze.marketing_campaigns"
)

campaigns = (
    bronze_campaigns
    .withColumn(
        "created_at",
        F.to_timestamp("created_at"),
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at"),
    )
    .withColumn(
        "start_date",
        F.to_date("start_date"),
    )
    .withColumn(
        "end_date",
        F.to_date("end_date"),
    )
)

silver_campaigns = latest_record(
    campaigns,
    "campaign_id",
).select(
    "campaign_id",
    "tenant_id",
    "campaign_name",
    "channel",
    "campaign_type",
    "objective",
    "status",
    "start_date",
    "end_date",
    "created_at",
    "updated_at",
)

(
    silver_campaigns.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.dim_campaign")
)


# ---------------------------------------------------------
# 2. Ad groups
# ---------------------------------------------------------

bronze_ad_groups = spark.table(
    "voyage.bronze.marketing_ad_groups"
)

ad_groups = (
    bronze_ad_groups
    .withColumn(
        "created_at",
        F.to_timestamp("created_at"),
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at"),
    )
)

silver_ad_groups = latest_record(
    ad_groups,
    "ad_group_id",
).select(
    "ad_group_id",
    "campaign_id",
    "tenant_id",
    "ad_group_name",
    "status",
    "created_at",
    "updated_at",
)

(
    silver_ad_groups.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.dim_ad_group")
)


# ---------------------------------------------------------
# 3. Ads
# ---------------------------------------------------------

bronze_ads = spark.table(
    "voyage.bronze.marketing_ads"
)

ads = (
    bronze_ads
    .withColumn(
        "created_at",
        F.to_timestamp("created_at"),
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at"),
    )
)

silver_ads = latest_record(
    ads,
    "ad_id",
).select(
    "ad_id",
    "ad_group_id",
    "campaign_id",
    "tenant_id",
    "ad_name",
    "creative_type",
    "status",
    "created_at",
    "updated_at",
)

(
    silver_ads.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.dim_ad")
)


# ---------------------------------------------------------
# 4. Campaign daily performance
# ---------------------------------------------------------

bronze_performance = spark.table(
    "voyage.bronze.marketing_campaign_performance"
)

performance = (
    bronze_performance
    .withColumn(
        "date",
        F.to_date("date"),
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at"),
    )
    .withColumn(
        "impressions",
        F.col("impressions").cast("bigint"),
    )
    .withColumn(
        "clicks",
        F.col("clicks").cast("bigint"),
    )
    .withColumn(
        "conversions",
        F.col("conversions").cast("bigint"),
    )
    .withColumn(
        "spend",
        F.col("spend").cast("decimal(14,2)"),
    )
    .withColumn(
        "revenue",
        F.col("revenue").cast("decimal(14,2)"),
    )
)

silver_performance = latest_record(
    performance,
    "metric_id",
).select(
    "metric_id",
    "date",
    "tenant_id",
    "campaign_id",
    "impressions",
    "clicks",
    "spend",
    "conversions",
    "revenue",
    "updated_at",
)

(
    silver_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.fact_campaign_daily")
)


print("Marketing Silver tables created successfully.")

#### Silver tables validation.

In [0]:
tables = [
    "voyage.silver.dim_campaign",
    "voyage.silver.dim_ad_group",
    "voyage.silver.dim_ad",
    "voyage.silver.fact_campaign_daily",
]

for table in tables:
    print(f"{table}: {spark.table(table).count()} rows")

#### checking of updated metric.

In [0]:
display(
    spark.sql("""
        SELECT
            metric_id,
            campaign_id,
            date,
            spend,
            updated_at
        FROM voyage.silver.fact_campaign_daily
        WHERE metric_id = 1
    """)
)

#### relationship between tables.

In [0]:
print(
    "Orphan ad groups:",
    spark.sql("""
        SELECT COUNT(*)
        FROM voyage.silver.dim_ad_group ag
        LEFT JOIN voyage.silver.dim_campaign c
            ON ag.campaign_id = c.campaign_id
        WHERE c.campaign_id IS NULL
    """).first()[0]
)

print(
    "Orphan ads:",
    spark.sql("""
        SELECT COUNT(*)
        FROM voyage.silver.dim_ad a
        LEFT JOIN voyage.silver.dim_ad_group ag
            ON a.ad_group_id = ag.ad_group_id
        WHERE ag.ad_group_id IS NULL
    """).first()[0]
)

print(
    "Orphan performance:",
    spark.sql("""
        SELECT COUNT(*)
        FROM voyage.silver.fact_campaign_daily p
        LEFT JOIN voyage.silver.dim_campaign c
            ON p.campaign_id = c.campaign_id
        WHERE c.campaign_id IS NULL
    """).first()[0]
)